# AI Repositories Showcase — Ranked by Stars & Freshness

A live-fetched leaderboard of ~25 influential open-source AI repositories across nine categories: LLM training, agent frameworks, inference/serving, RAG & vector DBs, computer vision, image generation, speech, RL, and foundation libraries.

Every star count and "last pushed" date below comes from the **GitHub REST API** at the moment this notebook was executed — no scraped lists, no stale hand-curation. If a row looks off, re-run the data cell.

The top 3 runnable repos are followed by a short verbatim snippet so you can see, not just be told, what each repo *does* in code.

## Table of Contents

1. [Curated repo list](#1-curated-repo-list) — the ~25 repos we track, grouped by category
2. [Fetch live GitHub metadata](#2-fetch-live-github-metadata) — one API call per repo
3. [Leaderboard: ranked by stars](#3-leaderboard-ranked-by-stars) — primary ranking
4. [Leaderboard: ranked by recency](#4-leaderboard-ranked-by-recency) — most recently pushed
5. [Featured snippet 1 — karpathy/micrograd](#5-featured-snippet-1--karpathymicrograd)
6. [Featured snippet 2 — rasbt/LLMs-from-scratch](#6-featured-snippet-2--rasbtllms-from-scratch)
7. [Featured snippet 3 — huggingface/transformers](#7-featured-snippet-3--huggingfacetransformers)
8. [Data sources](#8-data-sources)
9. [Build provenance](#9-build-provenance)


<a id="1-curated-repo-list"></a>
## 1. Curated repo list

These are the repos we will fetch. The list is intentionally cross-category — no point ranking only LLM repos against each other when "AI" today spans agents, diffusion, RL, and CV.

| Category | Repos |
|---|---|
| **LLM training / from-scratch** | `karpathy/nanoGPT`, `karpathy/llm.c`, `karpathy/micrograd`, `karpathy/minGPT`, `rasbt/LLMs-from-scratch`, `d2l-ai/d2l-en` |
| **Agent frameworks** | `langchain-ai/langchain`, `langchain-ai/langgraph`, `run-llama/llama_index`, `microsoft/autogen`, `crewAIInc/crewAI`, `huggingface/smolagents`, `pydantic/pydantic-ai` |
| **Inference / serving** | `vllm-project/vllm`, `ggerganov/llama.cpp`, `ollama/ollama` |
| **RAG / vector DB** | `chroma-core/chroma`, `weaviate/weaviate` |
| **Computer vision** | `ultralytics/ultralytics`, `facebookresearch/segment-anything` |
| **Image generation** | `AUTOMATIC1111/stable-diffusion-webui`, `comfyanonymous/ComfyUI`, `huggingface/diffusers` |
| **Speech** | `openai/whisper` |
| **Reinforcement learning** | `DLR-RM/stable-baselines3`, `Farama-Foundation/Gymnasium` |
| **Foundation libraries** | `huggingface/transformers`, `pytorch/pytorch`, `huggingface/datasets` |


<a id="2-fetch-live-github-metadata"></a>
## 2. Fetch live GitHub metadata

One unauthenticated GET to `https://api.github.com/repos/{owner}/{name}` per repo. Unauthenticated rate limit is 60/hour; we cap at ~30 repos.

The fields we keep:

- `stargazers_count` → ranking signal
- `pushed_at` → activity recency (default branch's last commit timestamp)
- `description` → one-line context
- `language` → primary language

Failures (network blip, repo moved) are surfaced as `error` rows so we never silently drop a repo.


In [1]:
import json
import time
import urllib.request
import urllib.error
from datetime import datetime, timezone

REPOS = [
    # LLM training / from-scratch
    ("karpathy/nanoGPT", "LLM training / from-scratch"),
    ("karpathy/llm.c", "LLM training / from-scratch"),
    ("karpathy/micrograd", "LLM training / from-scratch"),
    ("karpathy/minGPT", "LLM training / from-scratch"),
    ("rasbt/LLMs-from-scratch", "LLM training / from-scratch"),
    ("d2l-ai/d2l-en", "LLM training / from-scratch"),
    # Agent frameworks
    ("langchain-ai/langchain", "Agent frameworks"),
    ("langchain-ai/langgraph", "Agent frameworks"),
    ("run-llama/llama_index", "Agent frameworks"),
    ("microsoft/autogen", "Agent frameworks"),
    ("crewAIInc/crewAI", "Agent frameworks"),
    ("huggingface/smolagents", "Agent frameworks"),
    ("pydantic/pydantic-ai", "Agent frameworks"),
    # Inference / serving
    ("vllm-project/vllm", "Inference / serving"),
    ("ggerganov/llama.cpp", "Inference / serving"),
    ("ollama/ollama", "Inference / serving"),
    # RAG / vector DB
    ("chroma-core/chroma", "RAG / vector DB"),
    ("weaviate/weaviate", "RAG / vector DB"),
    # Computer vision
    ("ultralytics/ultralytics", "Computer vision"),
    ("facebookresearch/segment-anything", "Computer vision"),
    # Image generation
    ("AUTOMATIC1111/stable-diffusion-webui", "Image generation"),
    ("comfyanonymous/ComfyUI", "Image generation"),
    ("huggingface/diffusers", "Image generation"),
    # Speech
    ("openai/whisper", "Speech"),
    # Reinforcement learning
    ("DLR-RM/stable-baselines3", "Reinforcement learning"),
    ("Farama-Foundation/Gymnasium", "Reinforcement learning"),
    # Foundation libraries
    ("huggingface/transformers", "Foundation libraries"),
    ("pytorch/pytorch", "Foundation libraries"),
    ("huggingface/datasets", "Foundation libraries"),
]

UA = "abook-ai-showcase/1.0 (python-urllib; +https://github.com/abook)"


def fetch_repo(full_name: str) -> dict:
    url = f"https://api.github.com/repos/{full_name}"
    req = urllib.request.Request(
        url, headers={"User-Agent": UA, "Accept": "application/vnd.github+json"}
    )
    with urllib.request.urlopen(req, timeout=20) as r:
        return json.loads(r.read().decode("utf-8"))


rows = []
errors = []
for full_name, category in REPOS:
    try:
        data = fetch_repo(full_name)
        rows.append(
            {
                "repo": full_name,
                "category": category,
                "stars": data.get("stargazers_count", 0),
                "forks": data.get("forks_count", 0),
                "language": data.get("language") or "—",
                "pushed_at": data.get("pushed_at"),  # ISO-8601 UTC
                "description": (data.get("description") or "").strip(),
                "html_url": data.get("html_url"),
            }
        )
    except urllib.error.HTTPError as e:
        errors.append((full_name, f"HTTP {e.code} {e.reason}"))
    except Exception as e:
        errors.append((full_name, f"{type(e).__name__}: {e}"))
    time.sleep(0.15)  # be polite

print(f"fetched {len(rows)} repos; {len(errors)} errors")
for full_name, msg in errors:
    print(f"  ERROR  {full_name}  -> {msg}")
print(f"\nfetched at: {datetime.now(timezone.utc).isoformat(timespec='seconds')}")

fetched 29 repos; 0 errors

fetched at: 2026-05-26T19:47:32+00:00


<a id="3-leaderboard-ranked-by-stars"></a>
## 3. Leaderboard — ranked by stars

GitHub stars are a flawed but persistent popularity signal: easy to game in one direction (campaigns), impossible to fake at the scale of the top entries (>100k stars). For an overview, they correlate well enough with "what AI repos actually got adopted."

The `days_since_push` column is the antidote: a repo with 80k stars that hasn't been pushed in 3 years is doing very different work than one with 30k stars pushed yesterday.


In [3]:
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas"], check=True)
print("pandas installed")

pandas installed



[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip


pandas installed


In [4]:
import pandas as pd
from datetime import datetime, timezone

now = datetime.now(timezone.utc)

df = pd.DataFrame(rows)
df["pushed_dt"] = pd.to_datetime(df["pushed_at"], utc=True)
df["days_since_push"] = (now - df["pushed_dt"]).dt.days
df["stars_k"] = (df["stars"] / 1000).round(1)

by_stars = (
    df.sort_values("stars", ascending=False)
    .reset_index(drop=True)
    .assign(rank=lambda d: d.index + 1)[
        [
            "rank",
            "repo",
            "category",
            "stars_k",
            "language",
            "days_since_push",
            "description",
        ]
    ]
    .rename(columns={"stars_k": "stars (k)", "days_since_push": "days since push"})
)

# print as a clean fixed-width table the notebook will render nicely
pd.set_option("display.max_colwidth", 70)
pd.set_option("display.width", 200)
by_stars

,rank,repo,category,stars (k),language,days since push,description
0,1,ollama/ollama,Inference / serving,172.4,Go,1,"Get up and running with Kimi-K2.5, GLM-5, MiniMax, DeepSeek, gpt-o..."
1,2,AUTOMATIC1111/stable-diffusion-webui,Image generation,163.3,Python,85,Stable Diffusion web UI
2,3,huggingface/transformers,Foundation libraries,161.0,Python,0,🤗 Transformers: the model-definition framework for state-of-the-ar...
3,4,langchain-ai/langchain,Agent frameworks,137.7,Python,0,The agent engineering platform.
4,5,comfyanonymous/ComfyUI,Image generation,114.5,Python,0,"The most powerful and modular diffusion model GUI, api and backend..."
5,6,ggerganov/llama.cpp,Inference / serving,113.2,C++,0,LLM inference in C/C++
6,7,openai/whisper,Speech,100.6,Python,41,Robust Speech Recognition via Large-Scale Weak Supervision
7,8,pytorch/pytorch,Foundation libraries,100.2,Python,0,Tensors and Dynamic neural networks in Python with strong GPU acce...
8,9,rasbt/LLMs-from-scratch,LLM training / from-scratch,96.0,Jupyter Notebook,3,"Implement a ChatGPT-like LLM in PyTorch from scratch, step by step"
9,10,vllm-project/vllm,Inference / serving,81.1,Python,0,A high-throughput and memory-efficient inference and serving engin...


<a id="4-leaderboard-ranked-by-recency"></a>
## 4. Leaderboard — ranked by recency

Same data, sorted by `days_since_push` ascending. This view catches repos that aren't necessarily the most-starred but are *currently being worked on hardest* — typically a better leading indicator for "where is the field actually moving."


In [5]:
by_recency = (
    df.sort_values(["days_since_push", "stars"], ascending=[True, False])
    .reset_index(drop=True)
    .assign(rank=lambda d: d.index + 1)[
        ["rank", "repo", "category", "days_since_push", "stars_k", "pushed_at"]
    ]
    .rename(columns={"stars_k": "stars (k)", "days_since_push": "days since push"})
)
by_recency

,rank,repo,category,days since push,stars (k),pushed_at
0,1,huggingface/transformers,Foundation libraries,0,161.0,2026-05-26T19:37:53Z
1,2,langchain-ai/langchain,Agent frameworks,0,137.7,2026-05-26T19:00:42Z
2,3,comfyanonymous/ComfyUI,Image generation,0,114.5,2026-05-26T13:07:20Z
3,4,ggerganov/llama.cpp,Inference / serving,0,113.2,2026-05-26T18:43:31Z
4,5,pytorch/pytorch,Foundation libraries,0,100.2,2026-05-26T19:45:50Z
5,6,vllm-project/vllm,Inference / serving,0,81.1,2026-05-26T19:46:54Z
6,7,ultralytics/ultralytics,Computer vision,0,57.6,2026-05-26T17:24:37Z
7,8,crewAIInc/crewAI,Agent frameworks,0,52.2,2026-05-26T18:42:29Z
8,9,run-llama/llama_index,Agent frameworks,0,49.7,2026-05-26T17:50:15Z
9,10,huggingface/diffusers,Image generation,0,33.7,2026-05-26T15:48:38Z


<a id="5-featured-snippet-1--karpathymicrograd"></a>
## 5. Featured snippet 1 — `karpathy/micrograd`

A scalar-valued autograd engine in ~150 lines. Operator overloads (`+`, `*`, `**`, `relu`) build a DAG; `backward()` walks it in topological order, applying the chain rule. The code below is **verbatim from `micrograd/engine.py`**.


In [6]:
# Verbatim from karpathy/micrograd/blob/master/micrograd/engine.py
class Value:
    """stores a single scalar value and its gradient"""

    def __init__(self, data, _children=(), _op=""):
        self.data, self.grad = data, 0
        self._backward = lambda: None
        self._prev, self._op = set(_children), _op

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")

        def _backward():
            self.grad += out.grad
            other.grad += out.grad

        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad

        out._backward = _backward
        return out

    def __pow__(self, other):
        out = Value(self.data**other, (self,), f"**{other}")

        def _backward():
            self.grad += (other * self.data ** (other - 1)) * out.grad

        out._backward = _backward
        return out

    def relu(self):
        out = Value(0 if self.data < 0 else self.data, (self,), "ReLU")

        def _backward():
            self.grad += (out.data > 0) * out.grad

        out._backward = _backward
        return out

    def backward(self):
        topo, visited = [], set()

        def build(v):
            if v not in visited:
                visited.add(v)
                for c in v._prev:
                    build(c)
                topo.append(v)

        build(self)
        self.grad = 1
        for v in reversed(topo):
            v._backward()

    def __neg__(self):
        return self * -1

    def __sub__(self, o):
        return self + (-o)

    def __radd__(self, o):
        return self + o

    def __rmul__(self, o):
        return self * o

    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad})"


# The "the whole point" demo from nn-zero-to-hero lecture 1.
a, b = Value(-4.0), Value(2.0)
c = a + b
d = a * b + b**3
c = c + c + 1
c = c + 1 + c + (-a)
d = d + d * 2 + (b + a).relu()
d = d + 3 * d + (b - a).relu()
e = c - d
f = e**2
g = f * 0.5
g = g + 10.0 * f**-1

print(f"forward:  g.data = {g.data:.4f}    (expected 24.7041)")
g.backward()
print(f"dg/da = {a.grad:.4f}    (expected 138.8338)")
print(f"dg/db = {b.grad:.4f}    (expected 645.5773)")

forward:  g.data = 24.7041    (expected 24.7041)
dg/da = 138.8338    (expected 138.8338)
dg/db = 645.5773    (expected 645.5773)


<a id="6-featured-snippet-2--rasbtllms-from-scratch"></a>
## 6. Featured snippet 2 — `rasbt/LLMs-from-scratch`

The sliding-window dataset that LLM pretraining actually uses. The input/target pair is the *same token stream shifted by one* — that single fact is the entire next-token-prediction objective. Verbatim from `ch02/01_main-chapter-code/dataloader.ipynb`.


In [7]:
import subprocess, sys, importlib

for pkg in ("tiktoken", "torch"):
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

import urllib.request, pathlib, tiktoken, torch
from torch.utils.data import Dataset, DataLoader

# Real corpus (public domain) that Raschka's ch02 uses
url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"
path = pathlib.Path("the-verdict.txt")
if not path.exists():
    urllib.request.urlretrieve(url, path)
raw_text = path.read_text(encoding="utf-8")
print(f"corpus: {len(raw_text):,} chars, first 80: {raw_text[:80]!r}")


class GPTDatasetV1(Dataset):
    """Verbatim from rasbt/LLMs-from-scratch ch02/01_main-chapter-code/dataloader.ipynb."""

    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids, self.target_ids = [], []
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        for i in range(0, len(token_ids) - max_length, stride):
            self.input_ids.append(torch.tensor(token_ids[i : i + max_length]))
            self.target_ids.append(torch.tensor(token_ids[i + 1 : i + max_length + 1]))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


tokenizer = tiktoken.get_encoding("gpt2")
ds = GPTDatasetV1(raw_text, tokenizer, max_length=4, stride=4)
loader = DataLoader(ds, batch_size=8, shuffle=False, drop_last=True)

x, y = next(iter(loader))
print(f"\ninput  ids: {x[0].tolist()} -> {tokenizer.decode(x[0].tolist())!r}")
print(f"target ids: {y[0].tolist()} -> {tokenizer.decode(y[0].tolist())!r}")
print(f"\nbatch shapes: x={tuple(x.shape)}  y={tuple(y.shape)}")
print(f"sliding windows in 20k-char corpus: {len(ds)}")


[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip


corpus: 20,479 chars, first 80: 'I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow en'

input  ids: [40, 367, 2885, 1464] -> 'I HAD always'
target ids: [367, 2885, 1464, 1807] -> ' HAD always thought'

batch shapes: x=(8, 4)  y=(8, 4)
sliding windows in 20k-char corpus: 1286



[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip


<a id="7-featured-snippet-3--huggingfacetransformers"></a>
## 7. Featured snippet 3 — `huggingface/transformers`

The lingua franca model hub library. We avoid downloading weights (~500MB+ for even small models) and instead exercise the **tokenizer** — which is what most production code actually touches first. Compare GPT-2 BPE with BERT WordPiece on the same input to see how tokenization choices ripple through the rest of the stack.


In [8]:
import subprocess, sys, importlib

if importlib.util.find_spec("transformers") is None:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "transformers"], check=True
    )

from transformers import AutoTokenizer

sentence = "The Transformer is the most important architecture in modern deep learning."

# Tokenizer-only (no model weights). Both are small JSON / vocab.txt files (~1MB).
gpt2_tok = AutoTokenizer.from_pretrained("gpt2")
bert_tok = AutoTokenizer.from_pretrained("bert-base-uncased")

gpt2_ids = gpt2_tok.encode(sentence)
bert_ids = bert_tok.encode(sentence)

print(f"sentence: {sentence!r}\n")
print(f"GPT-2 BPE          : {len(gpt2_ids):2d} tokens  {gpt2_ids}")
print(f"  decoded pieces   : {[gpt2_tok.decode([t]) for t in gpt2_ids]}\n")
print(f"BERT WordPiece     : {len(bert_ids):2d} tokens  {bert_ids}")
print(f"  decoded pieces   : {bert_tok.convert_ids_to_tokens(bert_ids)}")


[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip
ownloads.


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

sentence: 'The Transformer is the most important architecture in modern deep learning.'

GPT-2 BPE          : 13 tokens  [464, 3602, 16354, 318, 262, 749, 1593, 10959, 287, 3660, 2769, 4673, 13]
  decoded pieces   : ['The', ' Trans', 'former', ' is', ' the', ' most', ' important', ' architecture', ' in', ' modern', ' deep
', ' learning', '.']

BERT WordPiece     : 15 tokens  [101, 1996, 10938, 2121, 2003, 1996, 2087, 2590, 4294, 1999, 2715, 2784, 4083, 1012, 102]
  decoded pieces   : ['[CLS]', 'the', 'transform', '##er', 'is', 'the', 'most', 'important', 'architecture', 'in', 'modern', 'de
ep', 'learning', '.', '[SEP]']


<a id="8-data-sources"></a>
## 8. Data sources

Every row in the leaderboard above came from a real GitHub API call at notebook-execution time.

| Source | URL pattern | Used in |
|---|---|---|
| GitHub REST API — repo metadata | `https://api.github.com/repos/{owner}/{name}` | section 2 (stars, `pushed_at`, description, language for all 29 repos) |
| `karpathy/micrograd` — `engine.py` | <https://github.com/karpathy/micrograd/blob/master/micrograd/engine.py> | section 5 (`Value` class verbatim + the canonical nn-zero-to-hero demo expression) |
| `rasbt/LLMs-from-scratch` — `the-verdict.txt` | <https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt> | section 6 (training corpus, downloaded into `notebooks/ai/the-verdict.txt`) |
| `rasbt/LLMs-from-scratch` — `dataloader.ipynb` | <https://github.com/rasbt/LLMs-from-scratch/blob/main/ch02/01_main-chapter-code/dataloader.ipynb> | section 6 (`GPTDatasetV1` verbatim) |
| Hugging Face Hub — `gpt2` & `bert-base-uncased` tokenizers | <https://huggingface.co/gpt2>, <https://huggingface.co/bert-base-uncased> | section 7 (vocab + merges only, ~2.5MB total; no model weights) |

No synthesized data, no hand-curated star counts. If a row looks stale, re-run section 2.


<a id="9-build-provenance"></a>
## 9. Build provenance

Authored by Claude Code via the **runt MCP server** (`runt mcp`) against a live Python 3.12 Jupyter kernel.

- Working directory: `/Users/mhuang/Documents/GitHub/abook/notebooks/ai/`
- Notebook: `ai-showcase.ipynb` (renamed from `dl-showcase.ipynb` on 2026-05-26)
- Build date: 2026-05-26
- Kernel: Python 3.12, uv-managed venv (`uv:pyproject`)
- Runtime-installed packages this session: `pandas`, `tiktoken`, `torch`, `transformers`
- All cells executed top-to-bottom in a single kernel session; outputs above are the actual run, not hand-typed.

Each `mcp__runt__*` tool call landed at the kernel in <100ms (transport stays open across the whole conversation), so the model was able to observe the `ModuleNotFoundError: No module named 'pandas'` in section 3 *as it happened* and `pip install` it in the next cell — instead of regenerating the whole `.ipynb` blind.
